# 🧩 كائنات المجال: `CurrencyPair` و `Interval`

---

## 💡 الفكرة المعمارية: لماذا ننشئ كائنات مخصصة للزوج والإطار الزمني؟

في التطبيقات البسيطة، يكتفي المطورون بتمرير النصوص كما هي (`pair: str`, `interval: str`). هذا يؤدي إلى مشاكل شائعة:

| المشكلة | الأثر |
|---------|-------|
| **تعدد الصيغ المدخلة** | المستخدم قد يكتب `eurusd` أو `EUR/USD` أو `eur/usd` → نفس الزوج يُخزَّن بأسماء مختلفة في قاعدة البيانات ويُرسل بشكل غير متوافق للـ API |
| **التحقق المبكر (Fail Early)** | التحقق من أن الزوج يتكون من عملة أساس (Base Currency) وعملة تسعير (Quote Currency) برموز ISO صالحة (3 أحرف لكل منهما) **قبل** استهلاك الـ API والاتصال بالشبكة |
| **الأطر الزمنية المعيارية** | منع تمرير قيم غير مدعومة (مثل `3days` أو `10m`) عبر `Enum` موحد يحدد الفواصل المسموح بها بدقة |

---

## 📊 مخطط العلاقة بين المكونات

```mermaid
graph TD
    A["User Input<br/>eurusd / EUR/USD / eur-usd"] --> B[CurrencyPair.from_string]
    C["User Input<br/>1h / 1d / 1m"] --> D[Interval.from_str]
    
    B --> E{Valid Format?}
    E -->|Yes| F["CurrencyPair<br/>base='EUR', quote='USD'"]
    E -->|No| G[ValidationError]
    
    D --> H{Alias Match?}
    H -->|Yes| I["Interval<br/>value='1h'"]
    H -->|No| J[ValueError]
    
    F --> K["symbol: 'EUR/USD'<br/>raw_symbol: 'EURUSD'"]
    I --> L["Canonical value<br/>'1day', '4h'"]
    
    K --> M[API + DB + Display]
    L --> M

    style B fill:#bbf,stroke:#333,stroke-width:2px
    style D fill:#bbf,stroke:#333,stroke-width:2px
    style F fill:#bfb,stroke:#333,stroke-width:2px
    style I fill:#bfb,stroke:#333,stroke-width:2px
    style G fill:#fdd,stroke:#333
    style J fill:#fdd,stroke:#333
```

**شرح المخطط:**

| المرحلة | الوصف |
|---------|-------|
| **Input** | أي صيغة يدخلها المستخدم (مع/بدون فاصل، حروف كبيرة/صغيرة) |
| **Factory Method** | `from_string` / `from_str` يحلل ويطبع المدخل |
| **Validation** | التحقق من ISO + الطول + التطابق |
| **Value Object** | كائن غير قابل للتعديل بعد الإنشاء |
| **Properties** | `symbol` / `raw_symbol` / `value` للاستخدام الموحد |
| **Downstream** | API + DB + Display — كلهم يستخدمون نفس الصيغة |

---

## 📊 دورة حياة المدخل

```mermaid
stateDiagram-v2
    [*] --> RawInput: "eurusd"
    RawInput --> Cleaned: strip + upper
    Cleaned --> Parsed: regex split or 6-char check
    Parsed --> Validated: ISO + length check
    Validated --> Object: CurrencyPair
    Object --> [*]: symbol = "EUR/USD"
    
    Cleaned --> Rejected: invalid format
    Validated --> Rejected: base == quote
    Rejected --> [*]: ValidationError
```

---

## 🔬 التشريح المعماري

### 1️⃣ `core/intervals.py` — الأطر الزمنية

**الفكرة:**
- كلاس `Interval` يرث من `str` + `Enum` في نفس الوقت.
- **وراثة `str`**: تسمح بمقارنته وتمريره **مباشرة كنص** في الـ API وقواعد البيانات — بدون `.value` إضافية.

**القيم المعيارية (9 أطر مدعومة):**

| الرمز | القيمة | المعنى |
|-------|--------|--------|
| `M1` | `"1min"` | دقيقة واحدة |
| `M5` | `"5min"` | 5 دقائق |
| `M15` | `"15min"` | 15 دقيقة |
| `M30` | `"30min"` | 30 دقيقة |
| `H1` | `"1h"` | ساعة |
| `H4` | `"4h"` | 4 ساعات |
| `D1` | `"1day"` | يوم |
| `W1` | `"1week"` | أسبوع |
| `M` | `"1month"` | شهر |

**دالة `from_str` — Factory Method:**

1. **تنظيف المدخل**: `strip()` + `lower()`
2. **قاموس Aliases** (خريطة اختصارات): يترجم الأشكال الشائعة مثل `1m`, `1d`, `day`, `1mo` → القيمة المعيارية
3. **لو المفتاح موجود في Aliases**: إرجاع `Interval` المطابق
4. **لو مش موجود**: رفع `ValueError` مع قائمة القيم المدعومة (Fail Fast)

**💡 تريكة التصميم:** قاموس الـ `aliases` داخل الدالة = **مرونة للمستخدم** بدون تلويث الـ Enum نفسه.

---

### 2️⃣ `core/pair.py` — زوج العملات

**الفكرة:**
- `@dataclass(frozen=True)` → كائن **غير قابل للتعديل** (Immutable).
- الحقول: `base` (عملة الأساس) + `quote` (عملة التسعير).

**دالة `__post_init__` — حارس البوابة (3 تحققات):**

1. **`base.isalpha() and len(base) == 3`** → عملة الأساس لازم ISO (3 أحرف)
2. **`quote.isalpha() and len(quote) == 3`** → نفس الشيء للتسعير
3. **`base != quote`** → يمنع `EUR/EUR` (غير منطقي مالياً)

**Properties (خصائص محسوبة):**

| الخاصية | القيمة | الاستخدام |
|---------|--------|-----------|
| `symbol` | `"EUR/USD"` | العرض والـ API |
| `raw_symbol` | `"EURUSD"` | بعض الـ APIs التي تطلب صيغة مدمجة |
| `__str__` | `"EUR/USD"` | الطباعة التلقائية |

**دالة `from_string` — Factory Method (3 حالات):**

| الحالة | المدخل | المعالجة |
|--------|--------|----------|
| **1. مع فاصل** | `EUR/USD`, `EUR-USD`, `EUR_USD`, `EUR USD` | `re.split(r"[/_\-\s]+")` → جزئين |
| **2. مدمج (6 أحرف)** | `EURUSD` | `cleaned[:3]` + `cleaned[3:]` |
| **3. غير صالح** | `INVALID`, `EURO` | `ValidationError` |

**💡 تريكة التصميم:** `re.split(r"[/_\-\s]+")` بتقبل **4 فواصل مختلفة** في سطر واحد — مرونة كاملة للمستخدم.

---

## 📊 مخطط القرار لـ `from_string`

```mermaid
flowchart TD
    A["eurusd / EUR/USD"] --> B[strip + upper]
    B --> C{Has separator?<br/>/ - _ space}
    C -->|Yes| D[Split into 2 parts]
    D --> E{Exactly 2 parts?}
    E -->|Yes| F[base + quote]
    E -->|No| H[ValidationError]
    C -->|No| G{Length == 6<br/>and isalpha?}
    G -->|Yes| I["base = [:3]<br/>quote = [3:]"]
    G -->|No| H
    F --> J[Validate ISO + base≠quote]
    I --> J
    J -->|Pass| K[CurrencyPair Object]
    J -->|Fail| H

    style A fill:#bbf,stroke:#333,stroke-width:2px
    style K fill:#bfb,stroke:#333,stroke-width:3px
    style H fill:#fdd,stroke:#333
```

---

## 🧪 التحقق من عمل المكونين

**سيناريوهات الاختبار المغطاة:**

| # | الاختبار | النتيجة المتوقعة |
|---|----------|-------------------|
| 1 | `from_string('eurusd')` | `EUR/USD` (Base: EUR, Quote: USD) |
| 2 | `from_string('GBP/usd')` | `GBP/USD` (توحيد الحالة) |
| 3 | `from_string('usd_jpy')` | `USD/JPY` (فاصل `_` مدعوم) |
| 4 | `from_str('1h')` | `"1h"` |
| 5 | `from_str('1d')` | `"1day"` (Alias → Canonical) |
| 6 | `from_string('INVALID')` | `ValidationError` (Fail Fast) |

**🎯 جوهر الاختبار:** إدخال **صيغ مختلفة** → **نفس الكائن الموحد** في النهاية. ده اللي بيضمن عدم تكرار البيانات في قاعدة البيانات.

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Canonical Form** | كل الأزواج تُخزَّن بصيغة `BASE/QUOTE` موحدة |
| **Fail Early** | الأخطاء تُرفض قبل استهلاك الـ API |
| **Type Safety** | بدل `str` عشوائي → `CurrencyPair` / `Interval` |
| **Immutability** | لا يمكن تعديل `base` أو `quote` بعد الإنشاء |
| **Single Source of Truth** | `symbol` و `raw_symbol` من مكان واحد |
| **User Flexibility** | دعم Aliases + فواصل متعددة |
| **Defensive Programming** | `__post_init__` يمنع البيانات التالفة |

---

## 🗺️ الموقع في خريطة الطريق

```mermaid
graph LR
    A["1. core/candle.py<br/>✅ مكتمل"] --> B["2. core/pair.py<br/>🆕 نحن هنا"]
    B --> C["3. core/intervals.py<br/>🆕 نحن هنا"]
    C --> D["4. storage/sqlite_storage.py<br/>تحديث: save_candles_bulk + interval"]
    D --> E["5. api/sources/twelvedata.py<br/>تحديث: دعم multi-pairs"]

    style A fill:#bfb,stroke:#333
    style B fill:#f9f,stroke:#333,stroke-width:3px
    style C fill:#f9f,stroke:#333,stroke-width:3px
    style D fill:#ddd,stroke:#333
    style E fill:#ddd,stroke:#333
```

---

## 📋 الخطوة التالية

> **الخطوة 2:** تحديث `SQLiteStorage` لدعم:
> - الحفظ الجماعي (`save_candles_bulk`)
> - إضافة حقل الفاصل الزمني `interval` للشموع

**السؤال الجوهري:** ليه `interval` لازم يكون في الشمعة نفسها؟

**الجواب:** لأن شمعة `1h` مختلفة تماماً عن شمعة `1day` **نفس الزوج وفي نفس التوقيت**. بدون تمييز `interval` في المفتاح الأساسي، البيانات هتتداخل وهنفقد القدرة على:
- استرجاع شموع إطار معين فقط.
- عمل Backtesting على إطارات مختلفة.
- مقارنة الاتجاهات على إطارات متعددة (Multi-timeframe analysis).

In [1]:
from core.pair import CurrencyPair
from core.intervals import Interval
from core.exceptions import ValidationError

# 1. فحص توحيد أشكال أزواج العملات المختلفة
p1 = CurrencyPair.from_string('eurusd')
p2 = CurrencyPair.from_string('GBP/usd')
p3 = CurrencyPair.from_string('usd_jpy')

print('Pairs Parsed:')
print(f'p1: {p1.symbol} (Base: {p1.base}, Quote: {p1.quote})')
print(f'p2: {p2.symbol}')
print(f'p3: {p3.symbol}')

# 2. فحص الأطر الزمنية
i1 = Interval.from_str('1h')
i2 = Interval.from_str('1d')
print('\nIntervals Parsed:')
print(f'i1: {i1.value}, i2: {i2.value}')

# 3. فحص الحماية من المدخلات الخاطئة
try:
    CurrencyPair.from_string('INVALID')
except ValidationError as e:
    print('\nExpected Validation Handled:', e)

Pairs Parsed:
p1: EUR/USD (Base: EUR, Quote: USD)
p2: GBP/USD
p3: USD/JPY

Intervals Parsed:
i1: 1h, i2: 1day

Expected Validation Handled: صيغة زوج العملات 'INVALID' غير صحيحة. أمثلة مقبولة: EUR/USD أو EURUSD


# 🗄️ Sprint 2 — الخطوة 2: ترقية طبقة التخزين (Bulk + Multi-Interval)

---

## 🎯 TL;DR

> **"من حفظ فردي إلى دفعات + دعم الأطر الزمنية"** — حلّ معضلتين: (1) تضارب التوقيت بين الإطارات، (2) بطء الكتابة. النتيجة: **~300x أسرع** و**صفر تداخل**.

---

## 💡 المعضلتان والحلول

### ⚠️ المعضلة 1: تضارب التوقيت (Interval Collision)

**Sprint 1:** `UNIQUE(pair, timestamp)` → شمعة `1day` و `4h` لنفس اليوم = **تداخل** (واحدة تمسح التانية).

**✅ الحل:** `UNIQUE(pair, interval, timestamp)` — المفتاح المنطقي أصبح **ثلاثياً**.

---

### ⚠️ المعضلة 2: عنق زجاجة الكتابة (I/O Bottleneck)

| الطريقة | 500 شمعة |
|---------|----------|
| `execute()` × 500 | 500 transaction → 🐌 ~15-30s |
| `executemany()` × 1 | 1 transaction → 🚀 ~50-100ms |

**✅ الحل:** `save_candles_bulk()` بـ `executemany` + **commit واحد**.

---

## 📊 مخطط القيد الفريد

```mermaid
erDiagram
    CANDLES {
        INTEGER id PK
        TEXT pair
        TEXT interval
        TEXT timestamp
        TEXT open
        TEXT high
        TEXT low
        TEXT close
        INTEGER volume
    }
    
    CANDLES ||--|| UNIQUE_KEY : "UNIQUE(pair, interval, timestamp)"
    
    UNIQUE_KEY {
        TEXT pair
        TEXT interval
        TEXT timestamp
    }
```

| السيناريو | النتيجة |
|-----------|---------|
| نفس الـ 3 قيم | 🔄 REPLACE |
| `interval` مختلف | ✅ صف جديد |
| `timestamp` مختلف | ✅ صف جديد |

---

## 🧩 المسؤوليات المحدثة

| # | المكوّن | التغيير |
|---|---------|---------|
| **1** | `Candle` | + `interval="1day"` (default) |
| **2** | `Storage ABC` | + `save_candles_bulk` (إلزامي) |
| **3** | `get_candles` | + `interval` parameter |
| **4** | `SQLiteStorage` | `executemany` + composite UNIQUE |
| **5** | `MemoryStorage` | Bulk + interval filter |

---

## 🔬 التشريح المعماري

### 1️⃣ `core/candle.py`

**التغيير:** `interval: str = "1day"` — **Backward Compatible**:
- Constructor قديم → يعمل
- `from_dict` → `data.get("interval", "1day")`
- `to_dict` → يُخرج `interval` دائماً

**تريكة `from_dict`:**
```python
raw_ts.replace(" ", "T")   # "2026-10-25 10:00" → "2026-10-25T10:00"
```
لأن `fromisoformat` يرفض المسافة في Python < 3.11.

---

### 2️⃣ `storage/base.py` — العقد

| الدالة | التغيير |
|--------|---------|
| `save_candle` | ✅ موجودة |
| **`save_candles_bulk`** | 🆕 **إلزامي** |
| `get_candles` | + `interval="1day"` |

**الفلسفة:** أي تطبيق جديد (Postgres, Redis) **يجب أن يفكر في الأداء من البداية** — لا "سأضيف Bulk لاحقاً".

---

### 3️⃣ `storage/sqlite_storage.py`

**`_init_db`:**
- ✅ + عمود `interval TEXT NOT NULL DEFAULT '1day'`
- ✅ `UNIQUE(pair, timestamp)` → `UNIQUE(pair, interval, timestamp)`
- ✅ فهرس `idx_pair_interval_timestamp`

**`save_candle`:** يُفوّض لـ `save_candles_bulk([c])` — **DRY**.

**`save_candles_bulk` — 7 خطوات:**

```mermaid
graph TD
    A[1. Guard: empty?] --> B[2. connect if needed]
    B --> C[3. INSERT OR REPLACE + :named]
    C --> D[4. [c.to_dict() for c in candles]]
    D --> E[5. executemany]
    E --> F[6. commit ONCE]
    F --> G[7. logger.info]

    style E fill:#bfb,stroke:#333,stroke-width:3px
```

**🎯 تريكة Named Placeholders:**
```python
# ✅ executemany يربط من dict تلقائياً
query = "INSERT ... VALUES (:pair, :interval, :timestamp, ...)"
records = [c.to_dict() for c in candles]
```
**لا ترتيب يدوي = صفر أخطاء**.

**`get_candles` — تريكة التحويل:**
```python
row_dict["datetime"] = row_dict.pop("timestamp")
```
لأن `Candle.from_dict` يبحث عن `"datetime"`، والعمود اسمه `"timestamp"`.

---

### 4️⃣ `storage/memory_storage.py`

| الدالة | التغيير |
|--------|---------|
| `save_candle` | ⬇️ يُفوّض للـ Bulk |
| `save_candles_bulk` | `self._candles.extend(candles)` |
| `get_candles` | filter `pair` + `interval` → `[-limit:]` |

**تريكة `hasattr(self, "_candles")`:** حماية لو لم تُهيَّأ في `__init__` — **Defensive**.

---

## 📊 دورة الحفظ الجماعي

```mermaid
sequenceDiagram
    participant S as Service
    participant St as Storage
    participant DB as SQLite

    S->>St: save_candles_bulk([c1..c500])
    St->>St: Guard + connect
    St->>St: records = [c.to_dict() ...]
    St->>DB: executemany (1 call)
    DB-->>St: OK
    St->>DB: commit (1)
    St-->>S: success
```

---

## 🧪 التحقق

| # | السيناريو | المتوقع |
|---|-----------|---------|
| 1 | 10 شموع `1h` في `:memory:` | ✅ Bulk |
| 2 | استرجاع 5 بـ `interval="1h"` | 5 شموع |
| 3 | `1h` + `1day` نفس التوقيت | **صفّان** |
| 4 | نفس الشمعة مرتين | **صف واحد** |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Composite Key** | صفر تداخل |
| **Bulk Insert** | ~300x أسرع |
| **Backward Compatible** | default `"1day"` |
| **DRY** | `save_candle` → `save_candles_bulk` |
| **Fail Safe** | `if not candles: return` |
| **Named Placeholders** | صفر أخطاء ترتيب |

---

## 📊 قبل / بعد

| السيناريو | ❌ Sprint 1 | ✅ Sprint 2 |
|-----------|------------|-------------|
| 500 شمعة | 500 txn | 1 txn (**~300x**) |
| `1h` + `1day` | تداخل | منفصلتان |
| فلترة إطار | مستحيل | `interval=` |
| العقد | `save_candle` | + `save_candles_bulk` |

---

## 🗺️ الموقع في Sprint 2

```mermaid
graph LR
    A["1. pair + intervals ✅"] --> B["2. Candle + Storage 🆕"]
    B --> C["3. TwelveData + Rate Limit"]
    C --> D["4. ForexService"]
    D --> E["5. CLI"]

    style B fill:#f9f,stroke:#333,stroke-width:3px
```

---

## 📋 الخطوة التالية

> **الخطوة 3:** `TwelveDataSource` + `ForexService` لدعم **Multi-pair** + **Rate Limiting**.

### 🤔 لماذا Rate Limiting؟

| المستوى | النتيجة |
|---------|---------|
| **تحت الحد** | ✅ 200 OK |
| **عند الحد** | ⚠️ 429 Too Many |
| **تجاوز مستمر** | 🚫 **حظر دائم** |

**الاستراتيجية:**
- **Throttling** — `sleep` بين الطلبات
- **Retry + Backoff** — عند 429
- **Batch** — عدة أزواج في استدعاء
- **Cache** — تجنب الطلبات المتكررة

In [3]:
from datetime import datetime, timedelta
from decimal import Decimal
from core.candle import Candle
from storage.sqlite_storage import SQLiteStorage

storage = SQLiteStorage(db_path=':memory:')

now = datetime.now()
candles_batch = [
    Candle(pair='GBP/USD', interval='1h', timestamp=now - timedelta(hours=i),
           open=Decimal('1.2500'), high=Decimal('1.2550'), low=Decimal('1.2480'),
           close=Decimal('1.2520'), volume=100)
    for i in range(10)
]

storage.save_candles_bulk(candles_batch)
results = storage.get_candles(pair='GBP/USD', interval='1h', limit=5)

print(f'Bulk Saved: {len(candles_batch)} | Retrieved: {len(results)}')
print(f'First Stored Interval: {results[0].interval} | Pair: {results[0].pair}')

[2026-10-26 23:38:25] [INFO] [forex_tracker]: Connected to SQLite database at: :memory:


[2026-10-26 23:38:25] [INFO] [forex_tracker]: Saved 10 candle(s) to SQLite.


Bulk Saved: 10 | Retrieved: 5
First Stored Interval: 1h | Pair: GBP/USD


In [4]:
print(results)

[Candle(pair='GBP/USD', timestamp=datetime.datetime(2026, 10, 26, 23, 38, 25, 644577), open=Decimal('1.2500'), high=Decimal('1.2550'), low=Decimal('1.2480'), close=Decimal('1.2520'), volume=100, interval='1h'), Candle(pair='GBP/USD', timestamp=datetime.datetime(2026, 10, 26, 22, 38, 25, 644577), open=Decimal('1.2500'), high=Decimal('1.2550'), low=Decimal('1.2480'), close=Decimal('1.2520'), volume=100, interval='1h'), Candle(pair='GBP/USD', timestamp=datetime.datetime(2026, 10, 26, 21, 38, 25, 644577), open=Decimal('1.2500'), high=Decimal('1.2550'), low=Decimal('1.2480'), close=Decimal('1.2520'), volume=100, interval='1h'), Candle(pair='GBP/USD', timestamp=datetime.datetime(2026, 10, 26, 20, 38, 25, 644577), open=Decimal('1.2500'), high=Decimal('1.2550'), low=Decimal('1.2480'), close=Decimal('1.2520'), volume=100, interval='1h'), Candle(pair='GBP/USD', timestamp=datetime.datetime(2026, 10, 26, 19, 38, 25, 644577), open=Decimal('1.2500'), high=Decimal('1.2550'), low=Decimal('1.2480'), cl

# 🔌 Sprint 2 — الخطوة 3: `TwelveDataSource` + `ForexService` (Multi-Pair + Rate Limiting)

---

## 🎯 TL;DR

> **"من زوج واحد إلى قائمة أزواج + حماية من الحظر"** — نضيف: تحقق مبكر (`CurrencyPair`/`Interval`)، حفظ دفعات، وتأخير بين الطلبات لتفادي `429`.

---

## 💡 المعضلتان والحلول

### ⚠️ 1. حدود الاستخدام (Rate Limits)

| الخطة | الحد |
|-------|------|
| TwelveData Free | **8 req/min** |

```mermaid
graph LR
    A[5 أزواج سريعاً] --> B{8 req/min?}
    B -- Over --> C[🚫 429 / Ban]
    B -- Throttled --> D[✅ 200 OK]

    style C fill:#f99,stroke:#333
    style D fill:#bfb,stroke:#333
```

**✅ الحل:** `time.sleep(rate_limit_delay)` بين الطلبات.

---

### ⚠️ 2. تكامل كائنات المجال

قبل الـ API: `CurrencyPair.from_string()` + `Interval.from_str()` → **Fail Early**.

---

## 📊 مخطط الطبقات

```mermaid
graph TD
    A["CLI: eurusd / 1h"] --> B[ForexService]
    B --> C["CurrencyPair + Interval"]
    C -->|Invalid| D[ValidationError]
    C -->|Valid| E[TwelveDataSource]
    E --> F[HttpClient + Throttle]
    F --> G[TwelveData API]
    G --> H[list Candle]
    H --> I[save_candles_bulk]
    I --> J[(SQLite)]

    style C fill:#bbf,stroke:#333,stroke-width:2px
    style I fill:#bfb,stroke:#333,stroke-width:2px
    style D fill:#fdd,stroke:#333
```

---

## 🧩 المسؤوليات المحدثة

| # | المكوّن | التغيير |
|---|---------|---------|
| **1** | `fetch_candles` | + `interval` param + يحقنه في كل `Candle` |
| **2** | `ForexService.__init__` | + `rate_limit_delay` |
| **3** | `fetch_and_store` | تحقق + حفظ bulk |
| **4** | **`fetch_and_store_multiple`** | 🆕 عدة أزواج + Throttle |
| **5** | **`get_watchlist`** | 🆕 نظرة سريعة |
| **6** | `get_history` | فلترة بـ `interval` |

---

## 🔬 التشريح المعماري

### 1️⃣ `TwelveDataSource.fetch_candles`

| الخطوة | الوصف |
|--------|-------|
| 1 | `params = {symbol, interval, outputsize, apikey}` |
| 2 | `http_client.get(...)` |
| 3 | `status == "error"` → `APIError` |
| 4 | `values` فاضية → `[]` (Warn) |
| 5 | **حقن** `item["symbol"]` + `item["interval"]` |
| 6 | `try/except ValidationError` → تخطي شمعة تالفة |

**🎯 تريكة الحقن:**
TwelveData لا يُرجع `symbol`/`interval` داخل كل `values` (فقط في `meta`) → نحقنها يدوياً قبل `Candle.from_dict`.

**🎯 تريكة Fail Soft:** شمعة تالفة → `logger.warning` + تخطي، **لا إسقاط الطلب**.

---

### 2️⃣ `ForexService` — 5 دوال

#### `__init__` — DI + Throttle
```python
(source=None, storage=None, rate_limit_delay=1.0)
```

| Parameter | الدور |
|-----------|-------|
| `source` / `storage` | DI للاختبار |
| `rate_limit_delay` | ثواني بين الطلبات |

---

#### `get_current_price` — أبسط دالة

`CurrencyPair.from_string` → `fetch_candles(limit=1)` → `candles[0].close`.

---

#### `fetch_and_store` — تحقق + bulk

```mermaid
graph LR
    A[raw] --> B[CurrencyPair]
    B --> C[Interval]
    C --> D[fetch]
    D --> E[save_bulk]

    style B fill:#bbf,stroke:#333
    style C fill:#bbf,stroke:#333
    style E fill:#bfb,stroke:#333
```

**التحقق المزدوج قبل استهلاك API.**

---

#### `fetch_and_store_multiple` — **جوهر Sprint 2**

```mermaid
sequenceDiagram
    participant S as Service
    participant API as TwelveData
    participant DB as Storage

    loop each pair
        S->>API: fetch_candles
        API-->>S: candles
        S->>DB: save_bulk
        Note over S: sleep (not last)
    end
```

| الخطوة | الوصف |
|--------|-------|
| 1 | Loop على `pairs` |
| 2 | `try: fetch_and_store(pair)` |
| 3 | `except: results[pair] = []` — **Fail Soft** |
| 4 | `sleep` **إلا للطلب الأخير** |
| 5 | return dict |

**🎯 تريكة لا-سليب-بعد-الأخير:** توفير ثانية كاملة في نهاية كل استدعاء.

**🎯 تريكة try/except لكل زوج:** زوج فاشل → البقية تكمل (**Isolation**).

---

#### `get_watchlist` — نظرة سريعة

| البند | `fetch_and_store_multiple` | `get_watchlist` |
|-------|---------------------------|------------------|
| يحفظ DB | ✅ | ❌ |
| limit | customizable | ثابت `1` |
| الإخراج | `dict[str, list[Candle]]` | `list[dict]` ملخص |

---

#### `get_history` — من DB فقط

`storage.get_candles(...)` — **لا يستهلك API quota**.

---

## 📊 مقارنة الطرق الخمس

| الطريقة | API؟ | DB؟ | Throttle؟ | الغرض |
|---------|------|-----|-----------|-------|
| `get_current_price` | ✅ | ❌ | ❌ | سعر لحظي |
| `fetch_and_store` | ✅ | ✅ | ❌ | جلب + حفظ |
| `fetch_and_store_multiple` | ✅ | ✅ | ✅ | عدة أزواج |
| `get_watchlist` | ✅ | ❌ | ✅ | نظرة سريعة |
| `get_history` | ❌ | ✅ | ❌ | من DB فقط |

---

## 🧪 التحقق

| # | الاختبار | المتوقع |
|---|----------|---------|
| 1 | `fetch_and_store('eurusd', '1h', 2)` | 2 شموع، `pair=EUR/USD`، `interval=1h` |
| 2 | `get_history('EUR/USD', '1h', 5)` | ≤ 5 شموع من DB |

**🎯 الجوهر:** المدخلات الخام (`eurusd`، `1h`) تُوحَّد **قبل** الـ API.

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Fail Early** | `CurrencyPair` + `Interval` قبل API |
| **Fail Soft** | تخطي شمعة/زوج تالف |
| **Throttling** | `rate_limit_delay` ضد 429 |
| **Isolation** | `try/except` لكل زوج |
| **Reuse** | multiple → single |
| **Optimization** | لا sleep بعد الأخير |
| **Injection** | `symbol`/`interval` تُحقن قبل `from_dict` |
| **Separation** | `get_history` لا يستهلك API |

---

## 📊 قبل / بعد

| السيناريو | ❌ Sprint 1 | ✅ Sprint 2 |
|-----------|------------|-------------|
| `eurusd` | خام للـ API | `EUR/USD` موحد |
| عدة أزواج | ❌ | ✅ `..._multiple` |
| حماية 429 | ❌ | ✅ Throttle |
| فلترة interval | ❌ | ✅ |
| نظرة سريعة | ❌ | ✅ `get_watchlist` |
| شمعة/زوج فاشل | يوقف الكل | Warn + يكمل |

---

## 🗺️ الموقع في Sprint 2

```mermaid
graph LR
    A["1. pair + intervals ✅"] --> B["2. Candle + Storage ✅"]
    B --> C["3. DataSource + Service 🆕"]
    C --> D["4. CLI: watchlist + multi-pair"]
    D --> E["5. Tests + Docs"]

    style C fill:#f9f,stroke:#333,stroke-width:3px
```

---

## 📋 الخطوة التالية

> **الخطوة 4:** CLI — `forex watchlist` + دعم تعدد الأزواج.

| الأمر | الوظيفة |
|-------|---------|
| `forex fetch --pairs EUR/USD GBP/USD` | عدة أزواج + Throttle |
| **`forex watchlist --pairs ...`** | 🆕 عرض سريع |
| `forex fetch --interval 1h` | تمرير الإطار |
| `forex list --interval 1h` | فلترة السجل |

**النمط:** نفس Dispatch — سطرين في `parser.py` + دالة في `commands.py`.

In [6]:
from storage.sqlite_storage import SQLiteStorage
storage.disconnect()

[2026-10-26 23:52:20] [INFO] [forex_tracker]: SQLite connection closed.


In [1]:
from services.forex_service import ForexService

service = ForexService(rate_limit_delay=0.5)

# 1. جلب شمعة بصيغة مدمجة (eurusd) وإطار زمني مختصر (1h)
candles = service.fetch_and_store(pair='eurusd', interval='1h', limit=2)
print(f'✅ Fetched {len(candles)} candles for: {candles[0].pair} | Interval: {candles[0].interval}')

# 2. استرجاع السجل بالإطار الزمني
history = service.get_history('EUR/USD', interval='1h', limit=5)
print(f'✅ History loaded: {len(history)} records')

[2026-10-26 23:53:24] [INFO] [forex_tracker]: Connected to SQLite database at: ./data\candles.db
[2026-10-26 23:53:24] [INFO] [forex_tracker]: جلب وتخزين 2 شمعة لزوج EUR/USD (1h)...
[2026-10-26 23:53:24] [INFO] [forex_tracker]: جلب 2 شمعة لزوج EUR/USD بإطار 1h من TwelveData...
[2026-10-26 23:53:24] [INFO] [forex_tracker]: إرسال طلب HTTP GET إلى: https://api.twelvedata.com/time_series
[2026-10-26 23:53:26] [INFO] [forex_tracker]: تم بنجاح استخراج 2 شمعة لزوج EUR/USD.
[2026-10-26 23:53:26] [INFO] [forex_tracker]: Saved 2 candle(s) to SQLite.


✅ Fetched 2 candles for: EUR/USD | Interval: 1h
✅ History loaded: 2 records


# 🖥️ Sprint 2 — الخطوة 4: تحديث الـ CLI (Multi-Pair + Watchlist)

---

## 🎯 TL;DR

> **"من أمر بزوج واحد إلى أوامر متعددة الأزواج"** — نضيف: `--interval` لـ `fetch`/`list`، `--pairs` لقائمة أزواج، وأمر جديد **`watchlist`** للمقارنة اللحظية.

---

## 💡 الفكرة المعمارية

| التحديث | الفائدة |
|---------|---------|
| **`--interval`** | جلب وعرض إطار زمني محدد |
| **`--pairs` (comma-separated)** | جلب عدة أزواج في أمر واحد |
| **`watchlist`** | جدول مقارنة لحظي بين الأزواج |

---

## 📊 مخطط تدفق الأوامر

```mermaid
graph TD
    A[forex] --> B{subcommand?}
    B -->|version| C[cmd_version]
    B -->|status| D[cmd_status]
    B -->|price| E[cmd_price]
    B -->|fetch| F[cmd_fetch]
    B -->|list| G[cmd_list]
    B -->|watchlist| H[cmd_watchlist]
    
    F --> I{pairs count}
    I -->|1| J[fetch_and_store]
    I -->|N| K[fetch_and_store_multiple]
    
    H --> L[get_watchlist]
    
    style F fill:#bbf,stroke:#333,stroke-width:2px
    style H fill:#bfb,stroke:#333,stroke-width:3px
    style K fill:#f9f,stroke:#333
```

---

## 🧩 المسؤوليات المحدثة

| # | المكوّن | التغيير |
|---|---------|---------|
| **1** | `cmd_fetch` | يدعم `--pairs` (comma) + `--interval` |
| **2** | `cmd_list` | + `--interval` |
| **3** | **`cmd_watchlist`** | 🆕 مقارنة لحظية |
| **4** | `parser.py` | تسجيل `watchlist` + `--pairs`/`--interval` |

---

## 🔬 التشريح المعماري

### 1️⃣ `cli/commands.py`

#### `cmd_fetch` — جوهر التحديث

```mermaid
graph LR
    A["--pairs EUR/USD,GBP/USD"] --> B[split(',')]
    B --> C{len == 1?}
    C -->|Yes| D[fetch_and_store]
    C -->|No| E[fetch_and_store_multiple]
    D --> F[print individual]
    E --> G[print batch table]

    style B fill:#bbf,stroke:#333
    style E fill:#bfb,stroke:#333,stroke-width:2px
```

**الخطوات:**

| الخطوة | الوصف |
|--------|-------|
| 1 | `raw_pairs = args.pairs or args.pair` — دعم `--pair` القديم |
| 2 | `pairs_list = [p.strip() for p in raw_pairs.split(",") if p.strip()]` |
| 3 | `if len == 1` → `fetch_and_store` (سلوك Sprint 1) |
| 4 | `else` → `fetch_and_store_multiple` (Batch) |
| 5 | طباعة النتائج (فردية أو جدول) |

**🎯 تريكة التوافق:** `getattr(args, "pairs", None) or getattr(args, "pair", "EUR/USD")` — يقبل الاسمين.

---

#### `cmd_list` — + `--interval`

| البند | Sprint 1 | Sprint 2 |
|-------|----------|----------|
| `--pair` | ✅ | ✅ |
| `--interval` | ❌ | 🆕 |
| `--limit` | ✅ | ✅ |

**Output:** جدول ثابت العرض بـ f-string (`{:<20}`) — لا يحتاج `rich`.

---

#### `cmd_watchlist` — **جديد**

| الخطوة | الوصف |
|--------|-------|
| 1 | `--pairs` افتراضي: `EUR/USD,GBP/USD,USD/JPY` |
| 2 | `ForexService(rate_limit_delay=1.0)` |
| 3 | `service.get_watchlist(pairs_list)` |
| 4 | طباعة جدول: Pair / Close / High / Low / Time |

**🎯 Output:** 5 أعمدة بجدول ثابت العرض.

---

### 2️⃣ `cli/parser.py` — تسجيل الأوامر

**الخيارات الجديدة:**

| الأمر | Options |
|-------|---------|
| `fetch` | `--pairs` (comma) + `--interval` + `--limit` |
| `list` | `--pair` + `--interval` + `--limit` |
| **`watchlist`** | 🆕 `--pairs` (comma) |

**النمط ثابت:** `add_parser(...)` → `add_argument(...)` → `set_defaults(func=...)`.

---

## 🧪 الاختبارات — إضافة `tests/test_multi_pairs.py`

**الفلسفة:** اختبار **بدون شبكة** — نستخدم `Mock` للـ `source` و `storage` حقيقي في `:memory:`.

### 📊 خريطة الاختبارات

```mermaid
graph TD
    A[TestMultiPairs] --> B[test_currency_pair_from_string]
    A --> C[test_interval_from_str]
    A --> D[test_fetch_and_store_multiple]
    A --> E[test_get_watchlist]
    A --> F[test_get_history_filters_interval]
    A --> G[test_bulk_save_no_collision]
    A --> H[test_throttling_applied]

    style A fill:#bbf,stroke:#333,stroke-width:3px
    style G fill:#bfb,stroke:#333,stroke-width:2px
```

### 🎯 الاختبارات المقترحة (وصف بدون كود)

| # | الاختبار | ما يتحقق منه |
|---|----------|---------------|
| **1** | `CurrencyPair.from_string` | `eurusd` / `EUR/USD` / `eur-usd` → `EUR/USD` موحد |
| **2** | `CurrencyPair` validation | `INVALID` → `ValidationError` |
| **3** | `Interval.from_str` | `1h` / `day` / `1d` → canonical |
| **4** | `Interval` invalid | `3days` → `ValueError` |
| **5** | `fetch_and_store_multiple` | 3 أزواج → 3 نتائج، كل زوج شموعه |
| **6** | `fetch_and_store_multiple` — partial failure | زوج يفشل → البقية تكمل |
| **7** | `get_watchlist` | list of dicts بـ `pair`, `close`, `high`, `low`, `time` |
| **8** | `get_history` — interval filter | `1h` و `1day` نفس التوقيت → صفّان منفصلان |
| **9** | Bulk save — no collision | حفظ `1h` + `1day` نفس الـ timestamp → صفّان |
| **10** | Bulk save — REPLACE | حفظ نفس الشمعة مرتين → صف واحد |
| **11** | Throttling | `rate_limit_delay=0.05` + 3 أزواج → زمن ≥ 0.1s |
| **12** | CLI parser — pairs parsing | `--pairs "EUR/USD,GBP/USD"` → list من 2 |

### 🎯 نمط الاختبار (Mock Strategy)

```mermaid
graph LR
    A[Test] --> B[MockSource]
    A --> C[SQLiteStorage :memory:]
    B --> D[returns fake candles]
    C --> E[real DB in RAM]
    A --> F[ForexService source=MockStorage=SQLite]

    style B fill:#bbf,stroke:#333
    style C fill:#bfb,stroke:#333
    style F fill:#f9f,stroke:#333,stroke-width:2px
```

**لماذا Mock؟**
- **سرعة**: لا شبكة.
- **حتمية**: نفس النتائج كل مرة.
- **عزل**: اختبار الخدمة، لا API.

**لماذا SQLite `:memory:`؟**
- يختبر **السلوك الحقيقي** للـ Storage (Unique, REPLACE, filters).
- **معزول** بين الاختبارات.

---

## 🧪 التحقق اليدوي — ملاحظات

| الأمر | ما يُثبته |
|-------|----------|
| `forex fetch --pairs EUR/USD --interval 1h --limit 3` | دعم `--interval` لزوج واحد |
| `forex list --pair EUR/USD --interval 1h` | فلترة السجل بإطار |
| `forex fetch --pairs EUR/USD,GBP/USD --interval 1day --limit 2` | Batch mode |
| `forex watchlist --pairs EUR/USD,GBP/USD` | جدول مقارنة |

---

## 🎯 القيمة المعمارية

| المبدأ | التطبيق |
|--------|---------|
| **Backward Compat** | `--pair` القديم يعمل |
| **Comma-separated** | `--pairs A,B,C` → list |
| **Branching** | `len == 1` → single، `>1` → batch |
| **Consistent UX** | نفس جدول العرض في كل الأوامر |
| **Throttle default** | `rate_limit_delay=1.0` |
| **Fail Fast** | `except ForexError` → stderr + exit(1) |

---

## 📊 قبل / بعد

| السيناريو | ❌ Sprint 1 | ✅ Sprint 2 |
|-----------|------------|-------------|
| عدة أزواج | ❌ | ✅ `--pairs A,B,C` |
| إطار زمني | ❌ | ✅ `--interval 1h` |
| `list` بإطار | ❌ | ✅ `--interval` |
| `watchlist` | ❌ | ✅ أمر جديد |
| اختبارات Sprint 2 | ❌ | ✅ 12 اختبار |

---

## 🗺️ الموقع في Sprint 2

```mermaid
graph LR
    A["1. pair + intervals ✅"] --> B["2. Candle + Storage ✅"]
    B --> C["3. DataSource + Service ✅"]
    C --> D["4. CLI + Tests 🆕"]
    D --> E["5. Docs + Close Sprint"]

    style D fill:#f9f,stroke:#333,stroke-width:3px
```

---

## 📋 الخطوة التالية

> **الخطوة 5 (الأخيرة في Sprint 2):** إغلاق Sprint بتوثيق كامل:
> - `docs/sprint_2_documentation.md`
> - تحديث `README.md` بالـ Roadmap
> - Tag: `v0.2.0`